In [2]:
import pyarrow.parquet as pq
import os 
import duckdb
import time

# User of Interests
Users who have blocked two people in their first week.

In [3]:
# Load the blocks database and profiles using DuckDB (memory efficient for large files)
blocks_path = "../data/user_activity/cleaned/blocks.parquet"
profiles_path = "../data/user_activity/cleaned/profiles.parquet"

# Connect to DuckDB
conn = duckdb.connect()

print("Analyzing blocks in first week after joining using DuckDB...")

# Get basic info about the data
total_blocks = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{blocks_path}')").fetchone()[0]
total_profiles = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{profiles_path}')").fetchone()[0]

print(f"Total blocks: {total_blocks:,}")
print(f"Total profiles: {total_profiles:,}")

# Query to find blocks in the first week after joining (days 0-6)
first_week_blocks_query = f"""
WITH blocks_with_join_days AS (
    SELECT 
        b.did_id,
        b.created_date AS block_date,
        p.created_date AS join_date,
        DATE_DIFF('day', p.created_date, b.created_date) AS days_since_join
    FROM read_parquet('{blocks_path}') b
    INNER JOIN read_parquet('{profiles_path}') p
        ON b.did_id = p.did_id
    WHERE DATE_DIFF('day', p.created_date, b.created_date) BETWEEN 0 AND 6
)
SELECT
    did_id,
    join_date,
    COUNT(*) AS block_count
FROM blocks_with_join_days
GROUP BY did_id, join_date
"""

# Execute query and get results
first_week_blocks = conn.execute(first_week_blocks_query).df()

print(f"\nUnique users who blocked someone in first week: {len(first_week_blocks):,}")
print(f"Total blocks in first week: {first_week_blocks['block_count'].sum():,}")

Analyzing blocks in first week after joining using DuckDB...
Total blocks: 75,122,543
Total profiles: 1,642,559



Unique users who blocked someone in first week: 413,315
Total blocks in first week: 4,258,817


In [4]:
# Save the set of users who blocked someone in their first week
first_week_blockers_path = "../data/user_activity/filtered/first_week_blockers.parquet"

num_samples = 100_000
first_week_blockers = (
    first_week_blocks[['did_id', 'join_date']]
    .drop_duplicates()
    .reset_index(drop=True)
)
sample_size = min(len(first_week_blockers), num_samples)
first_week_blockers = first_week_blockers.sample(n=sample_size, random_state=42).reset_index(drop=True)

first_week_blockers.to_parquet(first_week_blockers_path, index=False)

print(f"Saved {len(first_week_blockers):,} first-week blocker user ids to: {first_week_blockers_path}")

Saved 100,000 first-week blocker user ids to: ../data/user_activity/filtered/first_week_blockers.parquet


# Event Filtering
The criteria for filtering are:
- Events must involve at least one user of interest.
- Events must occur within the specified time window after the user's join date

In [5]:
def filter_events(input_path, output_path, db_name, query):

    print(f"Filtering {db_name} using DuckDB...")
    start_time = time.time()
    
    conn = duckdb.connect()
    
    # Register join_dat
    users_of_interests = conn.from_df(first_week_blockers)
    events_table = conn.read_parquet(input_path)
    
    # Get total row count before filtering
    total_rows = conn.execute("SELECT COUNT(*) FROM events_table").fetchall()[0][0]
    
    # Execute the provided query
    filtered_events = conn.execute(query).fetch_arrow_table()
    filtered_rows = filtered_events.num_rows
    
    # Write to parquet
    pq.write_table(filtered_events, output_path, compression='zstd')
    
    elapsed_time = time.time() - start_time
    input_size = os.path.getsize(input_path)
    output_size = os.path.getsize(output_path)
    retention_rate = (filtered_rows / total_rows * 100) if total_rows > 0 else 0
    
    print(f"\n✅ DuckDB filtering completed in {elapsed_time:.2f} seconds")
    print(f"Rows: {total_rows:,} → {filtered_rows:,} ({retention_rate:.1f}% kept)")
    print(f"File size: {input_size / (1024**3):.2f} GB → {output_size / (1024**3):.2f} GB")
    print(f"Output: {output_path}")
    
    conn.close()
    
    return {
        'elapsed_time': elapsed_time,
        'total_rows': total_rows,
        'filtered_rows': filtered_rows,
        'retention_rate': retention_rate,
        'input_size_gb': input_size / (1024**3),
        'output_size_gb': output_size / (1024**3)
    }
    


In [6]:
# Filter blocks with 14-day window (two-user events)
blocks_input_path = "../data/user_activity/cleaned/blocks.parquet"
blocks_output_path = "../data/user_activity/filtered/blocks.parquet"

blocks_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    -- Keep if did_id is in users AND within 14 days of joining
    ((uoi_act.did_id IS NOT NULL 
      AND DATE_DIFF('day', uoi_act.join_date, e.created_date) BETWEEN 0 AND 14)
    OR
    -- Keep if subject_did_id is in users AND within 14 days of joining
    (uoi_sub.did_id IS NOT NULL 
     AND DATE_DIFF('day', uoi_sub.join_date, e.created_date) BETWEEN 0 AND 14))
"""

stats_blocks = filter_events(blocks_input_path, blocks_output_path, "blocks", blocks_query)
print(f"Blocks filtered: {stats_blocks['filtered_rows']:,} blocks from {stats_blocks['total_rows']:,} total")

Filtering blocks using DuckDB...

✅ DuckDB filtering completed in 4.60 seconds
Rows: 75,122,543 → 2,199,771 (2.9% kept)
File size: 0.31 GB → 0.01 GB
Output: ../data/user_activity/filtered/blocks.parquet
Blocks filtered: 2,199,771 blocks from 75,122,543 total


In [7]:
# Filter posts with 7-day window (single-user events)
posts_input_path = "../data/user_activity/cleaned/posts.parquet"
posts_output_path = "../data/user_activity/filtered/posts.parquet"

posts_query = """
SELECT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi ON e.did_id = uoi.did_id
WHERE uoi.did_id IS NOT NULL
  AND DATE_DIFF('day', uoi.join_date, e.created_date) BETWEEN 0 AND 7
"""

stats_posts = filter_events(posts_input_path, posts_output_path, "posts", posts_query)
print(f"Posts filtered: {stats_posts['filtered_rows']:,} posts from {stats_posts['total_rows']:,} total")

Filtering posts using DuckDB...

✅ DuckDB filtering completed in 9.96 seconds
Rows: 734,849,771 → 5,175,728 (0.7% kept)
File size: 0.96 GB → 0.01 GB
Output: ../data/user_activity/filtered/posts.parquet
Posts filtered: 5,175,728 posts from 734,849,771 total


In [8]:
# Filter follows with 7-day window (two-user events)
follows_input_path = "../data/user_activity/cleaned/follows.parquet"
follows_output_path = "../data/user_activity/filtered/follows.parquet"

follows_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    -- Keep if did_id is in users AND within 7 days of joining
    ((uoi_act.did_id IS NOT NULL 
      AND DATE_DIFF('day', uoi_act.join_date, e.created_date) BETWEEN 0 AND 7)
    OR
    -- Keep if subject_did_id is in users AND within 7 days of joining
    (uoi_sub.did_id IS NOT NULL 
     AND DATE_DIFF('day', uoi_sub.join_date, e.created_date) BETWEEN 0 AND 7))
"""

stats_follows = filter_events(follows_input_path, follows_output_path, "follows", follows_query)
print(f"Follows filtered: {stats_follows['filtered_rows']:,} follows from {stats_follows['total_rows']:,} total")

Filtering follows using DuckDB...

✅ DuckDB filtering completed in 67.55 seconds
Rows: 973,455,738 → 37,510,213 (3.9% kept)
File size: 3.63 GB → 0.23 GB
Output: ../data/user_activity/filtered/follows.parquet
Follows filtered: 37,510,213 follows from 973,455,738 total


In [10]:
# Filter likes with 7-day window (two-user events)
likes_input_path = "../data/user_activity/cleaned/likes.parquet"
likes_output_path = "../data/user_activity/filtered/likes.parquet"

likes_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    ((uoi_act.did_id IS NOT NULL 
      AND DATE_DIFF('day', uoi_act.join_date, e.created_date) BETWEEN 0 AND 7)
    OR
    (uoi_sub.did_id IS NOT NULL 
     AND DATE_DIFF('day', uoi_sub.join_date, e.created_date) BETWEEN 0 AND 7))
"""

stats_likes = filter_events(likes_input_path, likes_output_path, "likes", likes_query)
print(f"Likes filtered: {stats_likes['filtered_rows']:,} likes from {stats_likes['total_rows']:,} total")

Filtering likes using DuckDB...

✅ DuckDB filtering completed in 117.11 seconds
Rows: 3,867,932,497 → 24,424,086 (0.6% kept)
File size: 9.12 GB → 0.16 GB
Output: ../data/user_activity/filtered/likes.parquet
Likes filtered: 24,424,086 likes from 3,867,932,497 total
